In [0]:
%sql

WITH silver_base AS (
  SELECT
    -- Business keys
    AccountName              AS account_name,
    ID                       AS owned_champion_id,

    -- Champion identity (not unique to owned copy)
    HeroID                   AS champion_id,

    -- Descriptors
    Name                     AS champion_name,
    Rank                     AS rank,
    Level                    AS level,
    EmpowerLevel             AS empower_level,
    Rarity                   AS rarity,
    Affinity                 AS affinity,
    Faction                  AS faction,

    -- Scroll usage
    UsedT1MasScrolls         AS used_t1_mastery_scrolls,
    UnUsedT1MasScrolls       AS unused_t1_mastery_scrolls,
    UsedT2MasScrolls         AS used_t2_mastery_scrolls,
    UnUsedT2MasScrolls       AS unused_t2_mastery_scrolls,
    UsedT3MasScrolls         AS used_t3_mastery_scrolls,
    UnUsedT3MasScrolls       AS unused_t3_mastery_scrolls,

    -- Stats
    HP                       AS hp,
    ATK                      AS atk,
    DEF                      AS def,
    CritRate                 AS crit_rate,
    CritDamage               AS crit_damage,
    SPD                      AS spd,
    ACC                      AS acc,
    RES                      AS res,

    -- Blessing
    BlessingID               AS blessing_id,
    BlessingGrade            AS blessing_grade,

    -- Lineage / audit (keep)
    run_id                   AS run_id,
    source_file              AS source_file,
    snapshot_ts              AS snapshot_ts,
    snapshot_date            AS snapshot_date,
    snapshot_version         AS snapshot_version,
    schema_version           AS schema_version

FROM workspace.raid.bronze_champindex
),
files AS (
    SELECT DISTINCT
        account_name,
        source_file,
        snapshot_date,
        snapshot_version
    FROM silver_base
),

latest_file AS (
    SELECT
        account_name,
        source_file,
        ROW_NUMBER() OVER (
            PARTITION BY account_name
            ORDER BY snapshot_date DESC, snapshot_version DESC
        ) AS rn
    FROM files
)

SELECT
    s.account_name,
    s.owned_champion_id,
    s.champion_id,
    s.champion_name,
    s.rank,
    s.level,
    s.empower_level,
    s.rarity,
    s.affinity,
    s.faction,

    s.used_t1_mastery_scrolls,
    s.unused_t1_mastery_scrolls,
    s.used_t2_mastery_scrolls,
    s.unused_t2_mastery_scrolls,
    s.used_t3_mastery_scrolls,
    s.unused_t3_mastery_scrolls,

    s.hp,
    s.atk,
    s.def,
    s.crit_rate,
    s.crit_damage,
    s.spd,
    s.acc,
    s.res,

    s.blessing_id,
    s.blessing_grade,

    s.run_id,
    s.source_file,
    s.snapshot_ts,
    s.snapshot_version,
    s.snapshot_date,
    s.schema_version

FROM silver_base s
JOIN latest_file l
  ON s.source_file = l.source_file
WHERE l.rn = 1
;

In [0]:
%sql
SELECT *
FROM raid.silver_champindex_current
WHERE account_name = 'Pega73'

In [0]:
%sql
WITH base AS (
    SELECT account_name, champion_id,
        SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS total_owned,
        SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS total_deleted,
        MAX(empower_level) AS max_empower_level,
        MAX(blessing_id) AS max_blessing_level,
        MAX(rank) AS max_rank
    FROM raid.silver_champindex_scd2
    GROUP BY account_name, champion_id
),

events_obtained AS (
    SELECT account_name, champion_id,
        COUNT(*) AS total_obtained
    FROM raid.silver_champindex_events_obtained
    GROUP BY account_name, champion_id
),

events_upgraded AS (
    SELECT account_name, champion_id,
        MAX(event_ts) AS last_upgraded_ts
    FROM raid.silver_champindex_events_upgrade
    GROUP BY account_name, champion_id
),

final AS (
    SELECT
        b.*,
        o.total_obtained,
        u.last_upgraded_ts,
        cfs.first_seen_ts,
        cl.champion_name,
        cl.rarity,
        cl.affinity,
        cl.faction
    FROM base b
    LEFT JOIN events_obtained o
        ON b.account_name = o.account_name
           AND b.champion_id = o.champion_id
    LEFT JOIN events_upgraded u
        ON b.account_name = u.account_name
           AND b.champion_id = u.champion_id
    JOIN raid.silver_champion_first_seen cfs
        ON b.account_name = cfs.account_name
           AND b.champion_id = cfs.champion_id
    JOIN raid.silver_champion_lookup cl
        ON b.champion_id = cl.champion_id
)

SELECT
    account_name,
    champion_id,
    champion_name,
    rarity,
    affinity,
    faction,
    first_seen_ts,
    last_upgraded_ts,
    total_owned,
    total_deleted,
    total_obtained,
    max_rank,
    max_empower_level,
    max_blessing_level
FROM final
;



In [0]:
%sql
SELECT *
FROM raid.silver_champion_lookup

In [0]:
%sql
--materialised
DROP TABLE IF EXISTS raid.silver_champion_lookup;
DROP TABLE IF EXISTS raid.silver_champindex_events_obtained;
DROP TABLE IF EXISTS raid.silver_champindex_events_upgrade;
DROP TABLE IF EXISTS raid.silver_champion_first_seen;
DROP TABLE IF EXISTS raid.silver_ops_processed_snapshots;
DROP TABLE IF EXISTS raid.snap_champindex;

--views
DROP VIEW IF EXISTS raid.silver_champindex_current;
DROP VIEW IF EXISTS raid.silver_ops_last_run_summary;
DROP VIEW IF EXISTS raid.silver_ops_out_of_order_snapshots;
DROP VIEW IF EXISTS raid.silver_ops_quarantine_champindex_classified;
DROP VIEW IF EXISTS raid.silver_champindex_scd2;
DROP VIEW IF EXISTS raid.silver_ops_quarantine_champindex_current_keys;
DROP VIEW IF EXISTS raid.silver_ops_quarantine_champindex_duplicates;
DROP VIEW IF EXISTS raid.silver_stg_champindex;
DROP VIEW IF EXISTS raid.silver_stg_champindex__classified;
DROP VIEW IF EXISTS raid.silver_stg_champindex__final;

In [0]:
%sql
SELECT * FROM raid.silver_champindex_scd2
LIMIT 10